# FYDP Active Learning: DeBERTa Entropy_AL Only

This notebook runs only the `entropy` acquisition arm. It keeps the same train/test protocol and output schema as the combined notebook, so its `round_metrics_entropy.csv` can be combined later with the matching other arm.


In [ ]:
# If running in Colab/Kaggle and packages are missing, uncomment this cell.
# !pip -q install transformers datasets accelerate evaluate sentencepiece scikit-learn pandas matplotlib seaborn

In [ ]:
import os
import re
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Configuration

This default matches the DeBERTa base notebook and methodology. RoBERTa and DistilBERT are optional extension runs only after the DeBERTa core result is complete.

In [ ]:
DATA_PATH = Path("/kaggle/input/datasets/armadarmard/fydpfinal/final_balanced_8K_curated.csv")
TEXT_COL = "Text"
LABEL_COL = "Label"

# Start with pilot. After it works, change this to "full".
RUN_MODE = "full"  # "pilot" or "full"

MODEL_PRESETS = {
    "deberta": "microsoft/deberta-v3-base",
    "roberta": "roberta-base",
    "distilbert": "distilbert-base-uncased",
}

# Main run should stay on DeBERTa because this matches your base notebook/methodology.
ACQUISITION_MODE = "entropy"
MODEL_KEY = "deberta"
MODEL_NAME = MODEL_PRESETS[MODEL_KEY]
MODEL_TAG = MODEL_NAME.replace("/", "_").replace("-", "_")

if RUN_MODE == "pilot":
    PILOT_ROWS = 500
    SEEDS = [42]
    AL_SEED_SIZE = 50
    AL_QUERY_SIZE = 25
    AL_ROUNDS = 3
    ROUND0_EPOCHS = 2
    EPOCHS = 1
else:
    PILOT_ROWS = None
    SEEDS = [42, 7, 123]
    AL_SEED_SIZE = 300
    AL_QUERY_SIZE = 100
    AL_ROUNDS = 15
    ROUND0_EPOCHS = 5
    EPOCHS = 3

# --- DeBERTa-v3 stability settings ---
# Round 0 gets longer warmup (0.3) than later rounds (0.1).
# All rounds use the same conservative LR to avoid collapse.
ROUND0_LEARNING_RATE = 1e-5
LEARNING_RATE = 1e-5
ROUND0_WARMUP_RATIO = 0.3
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 0.3
ADAM_EPSILON = 1e-6
FREEZE_EMBEDDINGS = True  # freezes embedding layer, key for DeBERTa stability
LR_DECAY = 0.88        # per-round LR multiplier to prevent collapse

MAX_LENGTH = 512
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.10
BATCH_SIZE = 8
WEIGHT_DECAY = 0.01

OUTPUT_ROOT = Path(f"outputs_{RUN_MODE}_{MODEL_TAG}_{ACQUISITION_MODE}_only")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "run_mode": RUN_MODE,
    "model": MODEL_NAME,
    "acquisition": ACQUISITION_MODE,
    "seeds": SEEDS,
    "rounds": AL_ROUNDS,
    "seed_size": AL_SEED_SIZE,
    "query_size": AL_QUERY_SIZE,
    "round0_epochs": ROUND0_EPOCHS,
    "later_round_epochs": EPOCHS,
})


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return classification_metrics(labels, preds)


class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

## Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df[[LABEL_COL, TEXT_COL]].dropna().reset_index(drop=True)
df[TEXT_COL] = df[TEXT_COL].apply(clean_text)

if PILOT_ROWS is not None:
    df, _ = train_test_split(
        df,
        train_size=PILOT_ROWS,
        stratify=df[LABEL_COL],
        random_state=42,
    )
    df = df.reset_index(drop=True)

encoder = LabelEncoder()
df["label_id"] = encoder.fit_transform(df[LABEL_COL])
label2id = {label: int(idx) for idx, label in enumerate(encoder.classes_)}
id2label = {int(idx): label for idx, label in enumerate(encoder.classes_)}
labels = [id2label[i] for i in range(len(id2label))]

with open(OUTPUT_ROOT / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label2id, f, indent=2)

print(df.shape)
print(df[LABEL_COL].value_counts().sort_index())
print(label2id)

## Acquisition Functions

In [ ]:
def get_entropy_indices(logits, query_size):
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    entropy = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    normalized_entropy = entropy / np.log(probs.shape[1])
    ranked = np.argsort(-normalized_entropy)
    return ranked[:query_size]


def get_random_indices(pool_size, query_size, seed):
    rng = np.random.default_rng(seed)
    return rng.choice(pool_size, size=min(query_size, pool_size), replace=False)


def aulc(results_df, metric="macro_f1"):
    values = []
    for (seed, acquisition), group in results_df.groupby(["seed", "acquisition"]):
        group = group.sort_values("labeled_size")
        area = np.trapezoid(group[metric], group["labeled_size"])
        values.append({"seed": seed, "acquisition": acquisition, f"aulc_{metric}": area})
    return pd.DataFrame(values)


## Train, Evaluate, and Predict Pool

In [ ]:
class NanGuardCallback(TrainerCallback):
    """Stops training the moment gradients or loss go non-finite, and prints
    which parameter first went bad so it is easy to locate in the logs."""
    def on_step_end(self, args, state, control, **kwargs):
        model = kwargs.get("model")
        if model is None:
            return control
        for name, param in model.named_parameters():
            if param.grad is not None and not torch.isfinite(param.grad).all():
                print(
                    f"[NanGuard] non-finite gradient at global_step={state.global_step} "
                    f"in parameter '{name}' -> stopping this round early."
                )
                control.should_training_stop = True
                return control
        return control


def freeze_embeddings(model):
    """Freezes the DeBERTa-v3 embedding layer (word + disentangled-attention
    position embeddings), which is the most commonly reported source of the
    NaN/gradient-blowup instability on small fine-tuning runs."""
    base_model = getattr(model, "deberta", None) or getattr(model, "base_model", None)
    if base_model is None or not hasattr(base_model, "embeddings"):
        print("[freeze_embeddings] could not locate embeddings module; skipping freeze.")
        return
    for param in base_model.embeddings.parameters():
        param.requires_grad = False


def make_training_args(output_dir, seed, epochs, round_id):
    lr = LEARNING_RATE * (LR_DECAY ** round_id)
    warmup_ratio = ROUND0_WARMUP_RATIO if round_id == 0 else WARMUP_RATIO
    return TrainingArguments(
        output_dir=str(output_dir),
        seed=seed,
        learning_rate=lr,
        adam_epsilon=ADAM_EPSILON,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=epochs,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=warmup_ratio,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        load_best_model_at_end=False,
        # metric_for_best_model removed
        # greater_is_better removed
        report_to="none",
        fp16=False,
        max_grad_norm=MAX_GRAD_NORM,
    )


def build_trainer(model_checkpoint, tokenizer, train_df, val_df, seed, output_dir, epochs, round_id):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    )
    if FREEZE_EMBEDDINGS:
        freeze_embeddings(model)

    train_ds = TextDataset(train_df[TEXT_COL], train_df["label_id"], tokenizer, MAX_LENGTH)
    val_ds = TextDataset(val_df[TEXT_COL], val_df["label_id"], tokenizer, MAX_LENGTH)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=make_training_args(output_dir, seed, epochs, round_id),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[
            NanGuardCallback(),
        ],
    )
    return trainer


def train_and_eval(model_checkpoint, tokenizer, train_df, val_df, test_df, seed, output_dir, epochs, round_id, return_probs=False):
    trainer = build_trainer(model_checkpoint, tokenizer, train_df, val_df, seed, output_dir, epochs, round_id)
    trainer.train()
    trainer.save_model()

    test_ds = TextDataset(test_df[TEXT_COL], test_df["label_id"], tokenizer, MAX_LENGTH)
    pred = trainer.predict(test_ds)
    if not np.isfinite(pred.predictions).all():
        raise ValueError(f"Non-finite test logits detected for checkpoint: {model_checkpoint}")

    y_true = pred.label_ids
    y_pred = np.argmax(pred.predictions, axis=-1)
    metrics = classification_metrics(y_true, y_pred)

    y_prob = None
    if return_probs:
        logits = pred.predictions
        y_prob = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return trainer, metrics, y_true, y_pred, y_prob


def predict_logits(trainer, tokenizer, pool_df):
    pool_ds = TextDataset(pool_df[TEXT_COL], pool_df["label_id"], tokenizer, MAX_LENGTH)
    pred = trainer.predict(pool_ds)
    if not np.isfinite(pred.predictions).all():
        raise ValueError("Non-finite pool logits detected during acquisition")
    return pred.predictions



## Active Learning Loop

In [ ]:
def stratified_initial_indices(train_pool_df, seed_size, seed):
    sampled, _ = train_test_split(
        train_pool_df,
        train_size=seed_size,
        stratify=train_pool_df["label_id"],
        random_state=seed,
    )
    return sampled.index.to_numpy()


def run_one_seed(seed, acquisition):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_val_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        stratify=df["label_id"],
        random_state=seed,
    )
    train_pool_df, val_df = train_test_split(
        train_val_df,
        test_size=VAL_SIZE_WITHIN_TRAIN,
        stratify=train_val_df["label_id"],
        random_state=seed,
    )

    train_pool_df = train_pool_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    initial_idx = stratified_initial_indices(train_pool_df, AL_SEED_SIZE, seed)
    labeled_df = train_pool_df.loc[initial_idx].copy().reset_index(drop=True)
    pool_df = train_pool_df.drop(index=initial_idx).copy().reset_index(drop=True)

    seed_results = []
    final_y_true = None
    return_probs = False
    final_y_pred = None
    current_checkpoint = MODEL_NAME
    consecutive_collapse = 0

    for round_id in range(AL_ROUNDS + 1):
        out_dir = OUTPUT_ROOT / f"seed_{seed}" / acquisition / f"round_{round_id}"
        round_epochs = ROUND0_EPOCHS if round_id == 0 else EPOCHS
        print(
            f"seed={seed} acquisition={acquisition} round={round_id} "
            f"labeled={len(labeled_df)} pool={len(pool_df)} epochs={round_epochs} "
            f"checkpoint={current_checkpoint}"
        )

        trainer, metrics, y_true, y_pred, y_prob = train_and_eval(
            current_checkpoint,
            tokenizer,
            labeled_df,
            val_df,
            test_df,
            seed + round_id,
            out_dir,
            round_epochs,
            round_id,
        )

        trainer.save_model(out_dir)
        tokenizer.save_pretrained(out_dir)
        current_checkpoint = str(out_dir)

        seed_results.append({
            "seed": seed,
            "acquisition": acquisition,
            "round": round_id,
            "labeled_size": len(labeled_df),
            "epochs": round_epochs,
            "checkpoint": current_checkpoint,
            **metrics,
        })

        # Collapse detection: skip to next seed after 2 consecutive collapsed rounds
        if metrics['macro_f1'] < 0.3:
            consecutive_collapse += 1
            if consecutive_collapse >= 2:
                print(f"  Collapse detected for 2 consecutive rounds (macro_f1={metrics['macro_f1']:.4f}). Skipping to next seed.")
                break
        else:
            consecutive_collapse = 0
        final_y_true, final_y_pred = y_true, y_pred
        if return_probs:
            final_probabilities[(seed, acquisition)] = y_prob

        if round_id == AL_ROUNDS or len(pool_df) == 0:
            return_probs = True
            break

        query_size = min(AL_QUERY_SIZE, len(pool_df))
        if acquisition == "entropy":
            logits = predict_logits(trainer, tokenizer, pool_df)
            selected_positions = get_entropy_indices(logits, query_size)
        elif acquisition == "random":
            selected_positions = get_random_indices(len(pool_df), query_size, seed + round_id)
        else:
            raise ValueError(f"Unknown acquisition: {acquisition}")

        selected_df = pool_df.iloc[selected_positions].copy()
        labeled_df = pd.concat([labeled_df, selected_df], ignore_index=True)
        pool_df = pool_df.drop(index=pool_df.index[selected_positions]).reset_index(drop=True)

        del trainer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(seed_results), final_y_true, final_y_pred

## Run Experiment

In [ ]:
all_results = []
final_probabilities = {}
final_predictions = {}

for seed in SEEDS:
    acquisition = ACQUISITION_MODE
    result_df, y_true, y_pred = run_one_seed(seed, acquisition)
    all_results.append(result_df)
    final_predictions[(seed, acquisition)] = (y_true, y_pred)

    seed_path = OUTPUT_ROOT / f"round_metrics_{ACQUISITION_MODE}_seed_{seed}.csv"
    result_df.to_csv(seed_path, index=False)
    print(f'  Seed {seed} results saved to {seed_path}')

results_df = pd.concat(all_results, ignore_index=True)
results_path = OUTPUT_ROOT / "round_metrics.csv"
results_acquisition_path = OUTPUT_ROOT / f"round_metrics_{ACQUISITION_MODE}.csv"
results_df.to_csv(results_path, index=False)
results_df.to_csv(results_acquisition_path, index=False)
print(f"Saved: {results_path}")
print(f"Saved: {results_acquisition_path}")
results_df


## Summaries

In [ ]:
summary = (
    results_df
    .groupby(["acquisition", "round", "labeled_size"])[["accuracy", "precision", "recall", "f1", "macro_f1", "weighted_f1"]]
    .agg(["mean", "std"])
    .reset_index()
)
summary_path = OUTPUT_ROOT / "summary_mean_std.csv"
summary_acquisition_path = OUTPUT_ROOT / f"summary_mean_std_{ACQUISITION_MODE}.csv"
summary.to_csv(summary_path, index=False)
summary.to_csv(summary_acquisition_path, index=False)
print(f"Saved: {summary_path}")
print(f"Saved: {summary_acquisition_path}")
summary


In [ ]:
aulc_df = aulc(results_df, metric="macro_f1")
aulc_summary = aulc_df.groupby("acquisition")["aulc_macro_f1"].agg(["mean", "std", "count"]).reset_index()
aulc_df.to_csv(OUTPUT_ROOT / "aulc_by_seed.csv", index=False)
aulc_summary.to_csv(OUTPUT_ROOT / "aulc_summary.csv", index=False)
aulc_df.to_csv(OUTPUT_ROOT / f"aulc_by_seed_{ACQUISITION_MODE}.csv", index=False)
aulc_summary.to_csv(OUTPUT_ROOT / f"aulc_summary_{ACQUISITION_MODE}.csv", index=False)
aulc_summary


## Learning Curves

Use Macro F1 for the learning curve because it treats all four classes equally. Weighted F1 is also saved in the CSV summaries to match the original DeBERTa methodology.

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=results_df,
    x="labeled_size",
    y="macro_f1",
    hue="acquisition",
    marker="o",
    errorbar="sd",
)
plt.title(f"{MODEL_NAME}: {ACQUISITION_MODE.title()} Learning Curve")
plt.xlabel("Labeled training samples")
plt.ylabel("Macro F1")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plot_path = OUTPUT_ROOT / f"learning_curve_macro_f1_{ACQUISITION_MODE}.png"
plt.savefig(plot_path, dpi=200)
plt.show()
plot_path


## Final Confusion Matrices

In [ ]:
for (seed, acquisition), (y_true, y_pred) in final_predictions.items():
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"Final confusion matrix: {acquisition}, seed {seed}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    cm_path = OUTPUT_ROOT / f"confusion_{acquisition}_seed_{seed}.png"
    plt.savefig(cm_path, dpi=200)
    plt.show()

    print(f"\nClassification report: acquisition={acquisition}, seed={seed}")
    print(classification_report(y_true, y_pred, target_names=labels))

# --- ROC Curves ---
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(y_true, y_prob, labels, seed, acquisition, output_root):
    y_bin = label_binarize(y_true, classes=range(len(labels)))
    n_classes = len(labels)
    plt.figure(figsize=(7, 6))
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], color=colors[i], lw=2,
                 label=f"{labels[i]} (AUC = {roc_auc[i]:.3f})")
    all_fpr = np.linspace(0, 1, 100)
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    roc_auc["macro"] = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, color="navy", lw=2, linestyle="--",
             label=f"Macro-average (AUC = {roc_auc['macro']:.3f})")
    plt.plot([0, 1], [0, 1], "k:", lw=1)
    plt.xlim([-0.02, 1.02]); plt.ylim([-0.02, 1.02])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves — {acquisition}, seed {seed}")
    plt.legend(loc="lower right", fontsize=8); plt.grid(alpha=0.3)
    plt.tight_layout()
    path = output_root / f"roc_{acquisition}_seed_{seed}.png"
    plt.savefig(path, dpi=200); plt.close()
    auc_df = pd.DataFrame([
        {"seed": seed, "acquisition": acquisition, "class": labels[i], "auc": roc_auc[i]}
        for i in range(n_classes)
    ] + [{"seed": seed, "acquisition": acquisition, "class": "macro", "auc": roc_auc["macro"]}])
    auc_path = output_root / f"auc_by_class_{acquisition}_seed_{seed}.csv"
    auc_df.to_csv(auc_path, index=False)
    return roc_auc

roc_aucs = {}
for (seed, acquisition), (y_true, y_pred) in final_predictions.items():
    y_prob = final_probabilities.get((seed, acquisition))
    if y_prob is not None:
        roc_aucs[(seed, acquisition)] = plot_roc_curves(
            y_true, y_prob, labels, seed, acquisition, OUTPUT_ROOT
        )

if roc_aucs:
    all_auc = []
    for (seed, acquisition), auc_dict in roc_aucs.items():
        for cls, val in auc_dict.items():
            all_auc.append({"seed": seed, "acquisition": acquisition, "class": cls, "auc": val})
    auc_all_df = pd.DataFrame(all_auc)
    auc_all_df.to_csv(OUTPUT_ROOT / "auc_summary.csv", index=False)
    print("ROC curves and AUC summary saved.")


## Combining Later

Run the entropy notebook and the random notebook separately. Each notebook writes the same schema:

```text
round_metrics_<acquisition>.csv
summary_mean_std_<acquisition>.csv
aulc_by_seed_<acquisition>.csv
aulc_summary_<acquisition>.csv
learning_curve_macro_f1_<acquisition>.png
confusion_<acquisition>_seed_<seed>.png
```

Later, put both output folders in the same workspace and run `combine_deberta_split_results.py` to produce the entropy-vs-random combined tables and plots.
